# 第 13 章 エンドツーエンドの実例

生データの前処理から、3 つのモデルの比較・評価までを 1 本のパイプラインに通します。

対応する記事: [第 13 章 エンドツーエンドの実例（Jupyter Notebook（Python） の言語版）](../../../docs/article/grokking-machine-learning/python/ch13.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch13_end_to_end import *

## 生データ

「40 歳未満かつ収入 400 超なら購入」という規則に従う擬似データを作ります。**9 行に 1 行は年齢が欠損** しています。実務のデータは、そのままではモデルに渡せません。

In [2]:
import random

rng = random.Random(1)
cities = ["tokyo", "osaka", "kyoto"]
rows = []
for i in range(40):
    age = rng.randint(18, 70)
    income = rng.randint(200, 900)
    rows.append({
        "age": "" if i % 9 == 0 else str(age),
        "income": str(income),
        "city": rng.choice(cities),
        "bought": "yes" if (age < 40 and income > 400) else "no",
    })

for row in rows[:5]:
    print(row)

{'age': '', 'income': '782', 'city': 'tokyo', 'bought': 'yes'}
{'age': '34', 'income': '320', 'city': 'osaka', 'bought': 'no'}
{'age': '66', 'income': '660', 'city': 'osaka', 'bought': 'no'}
{'age': '59', 'income': '588', 'city': 'tokyo', 'bought': 'no'}
{'age': '24', 'income': '699', 'city': 'tokyo', 'bought': 'yes'}


## 前処理

3 つの処理を通します。それぞれ「やらないとどうなるか」が明確です。

| 処理 | やらないとどうなるか |
| :--- | :--- |
| 欠損補完（中央値） | 学習が落ちる。平均だと 1 件の外れ値が全欠損を汚染する |
| 正規化 | 値の大きい特徴量（収入）だけが効く |
| One-Hot | カテゴリに存在しない大小関係が生まれる |

In [3]:
dataset = build_dataset(rows, "bought")

print("特徴量:", dataset.feature_names)
print(f"陽性 {sum(dataset.labels)} 件 / 全 {len(dataset.labels)} 件")
print()
for point, label in list(zip(dataset.points, dataset.labels))[:5]:
    print([f"{v:.3f}" for v in point], "→", label)

特徴量: ['age', 'income', 'city=kyoto', 'city=osaka', 'city=tokyo']
陽性 11 件 / 全 40 件

['0.686', '0.839', '0.000', '0.000', '1.000'] → 1
['0.314', '0.171', '0.000', '1.000', '0.000'] → 0
['0.941', '0.663', '0.000', '1.000', '0.000'] → 0
['0.804', '0.559', '0.000', '0.000', '1.000'] → 0
['0.118', '0.719', '0.000', '0.000', '1.000'] → 1


## 欠損補完と正規化を個別に確かめる

**平均ではなく中央値を使うのは、外れ値に引きずられないため** です。`[1, 2, 3, 1000]` の平均は 251.5 ですが、中央値は 2.5 です。

In [4]:
print("欠損補完:", impute_missing([1.0, 2.0, 3.0, 1000.0, None]))
print("正規化  :", normalize([10.0, 20.0, 30.0]))
print("定数列  :", normalize([5.0, 5.0, 5.0]), "← 0 除算しない")
print("One-Hot :", one_hot(["b", "a", "b"]))

欠損補完: [1.0, 2.0, 3.0, 1000.0, 2.5]
正規化  : [0.0, 0.5, 1.0]
定数列  : [0.0, 0.0, 0.0] ← 0 除算しない
One-Hot : ([[0.0, 1.0], [1.0, 0.0], [0.0, 1.0]], ['a', 'b'])


## 3 つのモデルを同じ土俵で比較する

第 6 章のロジスティック回帰、第 9 章の決定木、第 12 章の AdaBoost を、第 7 章の指標で評価します。

**正解率だけを見ていると差を見落とします。** 指標ごとに順位が変わりうることに注目してください。

In [5]:
evaluations = run_pipeline(rows)

print(f"{'モデル':<12} {'正解率':>8} {'適合率':>8} {'再現率':>8} {'F1':>8} {'AUC':>8}")
for e in evaluations:
    print(f"{e.name:<12} {e.accuracy:>8.3f} {e.precision:>8.3f} {e.recall:>8.3f} "
          f"{e.f1:>8.3f} {e.auc:>8.3f}")

print()
print("F1 が最良のモデル:", best_by_f1(evaluations).name)

モデル               正解率      適合率      再現率       F1      AUC
logistic        0.667    0.333    0.333    0.333    0.741
tree            0.750    0.500    1.000    0.667    0.833
adaboost        0.750    0.500    0.333    0.400    0.778

F1 が最良のモデル: tree


## 試してみる: 評価は揺れる

テストデータは 12 件しかありません。**1 件の当たり外れが正解率を 0.083 動かします。**

分割のシードを変えて、どれくらい結果が揺れるか見てみましょう。「モデル A のほうが 3% 良い」という報告が、この規模では意味を持たないことが分かります。実務では交差検証で複数の分割を試し、平均と分散を見ます。

In [6]:
print(f"{'シード':>6} {'テスト件数':>10} {'陽性数':>8}")
for seed in range(4):
    train_x, train_y, test_x, test_y = train_test_split(
        dataset.points, dataset.labels, test_ratio=0.3, seed=seed
    )
    print(f"{seed:>6} {len(test_y):>10} {sum(test_y):>8}")

   シード      テスト件数      陽性数
     0         12        3
     1         12        1
     2         12        5
     3         12        5
